### 1. Import Dependencies

In [1]:
import os
import pandas as pd #alias
import numpy as np #alias
import seaborn as sns
from matplotlib import pyplot as plt

In [3]:
print(os.listdir("processed"))

['ChurnModelling_Final.csv', 'ChurnModelling_Outliers_Handled.csv', 'ChurnModelling_Encoded.csv', 'ChurnModelling_Missing_values_handled.csv', 'ChrunModelling_Binning_Applied.csv', '.ipynb_checkpoints']


In [9]:
import os

print("Current folder:", os.getcwd())
print("Processed folder exists:", os.path.isdir("processed"))
print("Files here:", os.listdir("processed") if os.path.isdir("processed") else "Folder not found")

Current folder: /Users/kushalsharma/Documents/churn-prediction-data-pipeline-main
Processed folder exists: True
Files here: ['ChurnModelling_Final.csv', 'ChurnModelling_Outliers_Handled.csv', 'ChurnModelling_Encoded.csv', 'ChurnModelling_Missing_values_handled.csv', 'ChrunModelling_Binning_Applied.csv', '.ipynb_checkpoints']


In [11]:

file_path = os.path.abspath("processed/ChurnModelling_Binning_Applied.csv")
print(file_path)
print(os.path.exists(file_path)) 

/Users/kushalsharma/Documents/churn-prediction-data-pipeline-main/processed/ChurnModelling_Binning_Applied.csv
False


In [12]:
import glob

files = glob.glob("processed/*Binning*")

print(files)

['processed/ChrunModelling_Binning_Applied.csv']


| Variable     |   Preferred Encoding   | Why?                                                                 |
|--------------|------------------------|----------------------------------------------------------------------|
| Nominal      | One-Hot Encoding       | No inherent order -> Avoids implying false ordinal relationships     |
| Ordinal      | Label Encoding         | Preserves order -> Small integers represent increasing levels        |

- Nominal : means that there are no order. (ex: Gender, Eye Color, Type of Car, Hair Color, Political Party)
- Ordinal : means that there are some kind of a order. (ex: Educational Level, Income Level, Customer Satisfaction, Age Range)

In [13]:
df = pd.read_csv(files[0])
df.head()

,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,CreditScoreBins
0,France,Female,42.00,2,0.00,1,1,1,101348.88,1,Fair
1,Spain,Female,41.00,1,83807.86,1,0,1,112542.58,0,Fair
2,France,Female,42.00,8,159660.80,3,1,0,113931.57,1,Poor
3,France,Female,38.91,1,0.00,2,0,0,93826.63,0,Good
4,Spain,Female,43.00,2,125510.82,1,1,1,79084.10,0,Excellent


`Gender <Male, Female>`   // we will create 'n'  number of columns based on the number of diffrent columns.

|          |  Gender_Male | Gender_Female  |
|----------|--------------|----------------|
| Male     |       1      |       0        |
| Female   |       0      |       1        |

this is how the one-hot encoding representation usually works like.

`(Assume Gender is Ordinal);`
Male -> 1
Female -> 0


In this dataset these are the Values representations:

- Gender -> Nominal
- Geography -> Nominal
- CreditScoreBins -> Ordinal

In [10]:
df = pd.read_csv("processed/ChurnModelling_Binning_Applied.csv")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'processed/ChurnModelling_Binning_Applied.csv'

### 2. Encode Nominal Variables

In [14]:
nominal_variables = ['Geography', 'Gender']   #putting them in a list

# Create dummy variables
geography_dummies = pd.get_dummies(df['Geography'], prefix="Geography")
gender_dummies = pd.get_dummies(df['Gender'], prefix="Gender")

# Start with original df
df_encoded = df.copy()

# Add dummy variables
df_encoded = pd.concat([df_encoded, geography_dummies, gender_dummies], axis=1)

# Drop original categorical columns
df_encoded.drop(['Geography', 'Gender'], axis=1, inplace=True)

df_encoded


,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,CreditScoreBins,Geography_France,Geography_Germany,Geography_Spain,Gender_Female,Gender_Male
0,42.00,2,0.00,1,1,1,101348.88,1,Fair,True,False,False,True,False
1,41.00,1,83807.86,1,0,1,112542.58,0,Fair,False,False,True,True,False
2,42.00,8,159660.80,3,1,0,113931.57,1,Poor,True,False,False,True,False
3,38.91,1,0.00,2,0,0,93826.63,0,Good,True,False,False,True,False
4,43.00,2,125510.82,1,1,1,79084.10,0,Excellent,False,False,True,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,39.00,5,0.00,2,1,0,96270.64,0,Very Good,True,False,False,False,True
9996,35.00,10,57369.61,1,1,1,101699.77,0,Poor,True,False,False,False,True
9997,36.00,7,0.00,1,0,1,42085.58,1,Good,True,False,False,True,False
9998,42.00,3,75075.31,2,1,0,92888.52,1,Very Good,False,True,False,False,True


### 3. Encode Ordinal Variables

In [15]:
encode_dict_creditscore = {
                                'Poor'      : 0,
                                'Fair'      : 1,
                                'Good'      : 2,
                                'Very Good' : 3,
                                'Excellent' : 4
                            }

df_encoded['CreditScoreBins'] = df_encoded['CreditScoreBins'].map(encode_dict_creditscore)
df_encoded

,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,CreditScoreBins,Geography_France,Geography_Germany,Geography_Spain,Gender_Female,Gender_Male
0,42.00,2,0.00,1,1,1,101348.88,1,1,True,False,False,True,False
1,41.00,1,83807.86,1,0,1,112542.58,0,1,False,False,True,True,False
2,42.00,8,159660.80,3,1,0,113931.57,1,0,True,False,False,True,False
3,38.91,1,0.00,2,0,0,93826.63,0,2,True,False,False,True,False
4,43.00,2,125510.82,1,1,1,79084.10,0,4,False,False,True,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,39.00,5,0.00,2,1,0,96270.64,0,3,True,False,False,False,True
9996,35.00,10,57369.61,1,1,1,101699.77,0,0,True,False,False,False,True
9997,36.00,7,0.00,1,0,1,42085.58,1,2,True,False,False,True,False
9998,42.00,3,75075.31,2,1,0,92888.52,1,3,False,True,False,False,True


- Now all of them are in numerical formats, but in different ranges.
- We need to fix that.

In [20]:
df_encoded.to_csv('processed/ChurnModelling_Encoded.csv', index=False)

In [21]:
import pandas as pd
import glob

files = glob.glob("processed/*Binning*")
print(files)

['processed/ChrunModelling_Binning_Applied.csv']


In [22]:
df = pd.read_csv(files[0])
df.head()

,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,CreditScoreBins
0,France,Female,42.00,2,0.00,1,1,1,101348.88,1,Fair
1,Spain,Female,41.00,1,83807.86,1,0,1,112542.58,0,Fair
2,France,Female,42.00,8,159660.80,3,1,0,113931.57,1,Poor
3,France,Female,38.91,1,0.00,2,0,0,93826.63,0,Good
4,Spain,Female,43.00,2,125510.82,1,1,1,79084.10,0,Excellent


### Also you can do these with Scikit-Learn as well.

In [24]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

ohe_geography = OneHotEncoder()
ohe_gender = OneHotEncoder()
le_credit_score = LabelEncoder()

ohe_geography.fit(df[['Geography']])
ohe_gender.fit(df[['Gender']])
le_credit_score.fit(df['CreditScoreBins'])

Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)Holds the label for each class.","ndarray[object](5,)","['Excellent','Fair','Good','Poor','Very Good']"


In [27]:
gender_ohe = ohe_gender.transform(
    df['Gender'].to_numpy().reshape(-1, 1)
).toarray()

gender_ohe


/opt/anaconda3/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


array([[1., 0.],
       [1., 0.],
       [1., 0.],
       ...,
       [1., 0.],
       [0., 1.],
       [1., 0.]], shape=(10000, 2))

In [29]:
gender_ohe = ohe_gender.transform(
    df['Gender'].to_numpy().reshape(-1, 1)
).toarray()

gender_ohe

/opt/anaconda3/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


array([[1., 0.],
       [1., 0.],
       [1., 0.],
       ...,
       [1., 0.],
       [0., 1.],
       [1., 0.]], shape=(10000, 2))